In [1]:
# 환경 점검 — 이 셀이 "환경 준비 완료"를 출력해야 다음 셀로 진행합니다
from dotenv import load_dotenv
import os

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), (
    "❌ OPENAI_API_KEY 없음 — .env 파일 확인 (📖 강의교안 모듈 1-4 참조)"
)
if not os.getenv("LANGCHAIN_API_KEY"):
    print("⚠️  LANGCHAIN_API_KEY 없음 — LangSmith 트레이스가 기록되지 않습니다")

# Tavily는 선택 — 없어도 Step 1~4 실습 가능, Step 5에서 필요
TAVILY_AVAILABLE = bool(os.getenv("TAVILY_API_KEY"))

print("✅ 환경 준비 완료")
print(f"   API 키 앞 7자: {os.getenv('OPENAI_API_KEY')[:7]}...")
print(f"   Tavily 키   : {'✅ 있음' if TAVILY_AVAILABLE else '⚠️  없음 (Step 5에서 발급 안내)'}")

✅ 환경 준비 완료
   API 키 앞 7자: sk-proj...
   Tavily 키   : ✅ 있음


In [2]:
# LLM 공통 초기화 — 이후 모든 Step에서 재사용합니다
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ LLM 초기화 완료:", llm.model_name)

✅ LLM 초기화 완료: gpt-4o-mini


In [3]:
@tool
def get_employee_info(employee_id: str) -> dict:
    """
    직원 정보를 DB에서 조회합니다.

    사용 시점: 특정 직원의 이름, 부서, 직급 정보가 필요할 때.
    주의: 직원 ID는 반드시 'EMP' + 3자리 숫자 형식 (예: EMP001)

    Args:
        employee_id: 직원 고유 ID (형식: EMP + 3자리 숫자)
    Returns:
        직원 정보 dict: name, dept, level 키 포함
    """
    db = {
        "EMP001": {"name": "김철수", "dept": "AI개발팀",  "level": "팀장"},
        "EMP002": {"name": "이영희", "dept": "기획팀",    "level": "PM"},
        "EMP003": {"name": "박민준", "dept": "데이터팀",  "level": "시니어"},
    }
    return db.get(employee_id, {"error": f"직원 없음: {employee_id}"})

In [4]:
@tool
def calculate(expression: str) -> str:
    """
    수학 계산을 수행합니다.
    사용 시점: 사칙연산 등 숫자 계산이 필요할 때.
    expression: 계산 가능한 수식 (예: '1234 * 567', '100 / 4')
    """
    # ⚠️ 교육용 코드: eval()은 프로덕션에서 사용 금지 (보안 취약점)
    try: return str(eval(expression))
    except Exception as e: return f"계산 오류: {str(e)}"

@tool
def get_weather(city: str) -> str:
    """
    도시 날씨를 조회합니다.
    사용 시점: 특정 도시의 현재 날씨·기온을 물어볼 때.
    예시 질문: '서울 날씨', '오늘 도쿄 기온', '부산 날씨 알려줘'
    """
    return f"{city}: 맑음, 39도"  # Mock 데이터

In [5]:
from langchain_tavily import TavilySearch as _TS
@tool
def web_search(query: str, max_results: int = 5) -> str:
    """
    웹을 실시간으로 검색해 최신 정보를 가져옵니다.

    사용 시점: LLM 학습 데이터 이후 최신 정보가 필요할 때.
    사용하지 말 것: 역사적 사실, 일반 상식.

    Args:
        query: 검색어 (한국어/영어 모두 가능)
        max_results: 반환할 결과 수 (기본 5, 최대 20)
    """
    results = _TS(max_results=max_results).invoke(query)["results"]
    return "\n---\n".join(
        f"제목: {d.get('title','N/A')}\nURL: {d.get('url','')}\n내용: {d.get('content','')}"
        for d in results
    )

In [6]:
from langchain_tavily import TavilySearch as _TS2

@tool
def news_search(query: str, max_results: int = 5) -> str:
    """
    최신 뉴스 기사를 검색합니다.

    사용 시점: 오늘 뉴스, 최근 사건·사고·정책 발표 등 시사 정보가 필요할 때.
    사용하지 말 것: 역사적 사실이나 웹 일반 검색으로 충분한 질문.

    Args:
        query: 뉴스 검색어
        max_results: 결과 수 (기본 5)
    """
    results = _TS2(max_results=max_results, topic="news").invoke(query)["results"]
    return "\n---\n".join(
        f"제목: {d.get('title','N/A')}\nURL: {d.get('url','')}\n내용: {d.get('content','')[:___]}"
        for d in results
    )

In [7]:
from datetime import datetime
@tool
def current_date() -> str:
    """
    현재 날짜를 조회합니다.

    사용 시점: 오늘 날짜·현재 시각이 필요할 때.
    예시 질문: '오늘 몇 월이야?', '지금 몇 년도야?', '이번 달이 뭐야?'

    Returns:
        str: YYYY-MM-DD 형식의 현재 날짜
    """
    return datetime.now().strftime("%Y-%m-%d")

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import BaseMessage

# ── 시스템 프롬프트 ────────────────────────────────────────────────────
SYSTEM_CONTENT = "너는 한국의 예의바른 교사야. 짧고 구조적으로 대답해줘."

# ── LCEL 프롬프트 ──────────────────────────────────────────────────────
# {input}: 문자열 1개를 받는 자리 / MessagesPlaceholder("chat_history"): 메시지 리스트 전체를 받는 자리
prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_CONTENT),
    MessagesPlaceholder("chat_history"),   # ← 이전 대화 목록 삽입 위치
    ("human", "{input}"),
])

# ── 파서 ───────────────────────────────────────────────────────────────
parser = StrOutputParser()

# ── 세션별 히스토리 저장소 ─────────────────────────────────────────────
# 실무에서는 dict 대신 DB(Redis, PostgreSQL) 사용
memory_store: dict = {}

def get_history(session_id: str) -> InMemoryChatMessageHistory:
    """session_id에 해당하는 이력 반환 (없으면 새로 생성)"""
    return memory_store.setdefault(session_id, InMemoryChatMessageHistory())

def clear_history(session_id: str) -> None:
    """특정 세션의 대화 이력 초기화"""
    get_history(session_id).clear()


def _build_agent_runnable(llm, tools: list) -> RunnableLambda:
    """
    prompt → model → 도구 호출 루프를 하나의 Runnable로 감쌉니다.

    RunnableWithMessageHistory는 반환값이 BaseMessage의 리스트이면 "각 메시지를
    있는 그대로" history에 저장합니다(문자열이면 AIMessage 하나로 감싸서 저장하는
    것과 달리). 그래서 tool_calls가 담긴 AIMessage와 ToolMessage까지 전부
    이력에 영구히 남길 수 있습니다 — 질문하신 "AI뿐 아니라 tool_calls·tool
    메시지도 history에 저장" 요구사항이 이 반환 형태로 해결됩니다.
    """
    tool_list = {t.name: t for t in tools}
    llm_wt = llm.bind_tools(tools)

    def _run(payload: dict) -> list[BaseMessage]:
        # ① prompt: 시스템 + chat_history + 이번 입력 → 모델 호출용 메시지 리스트
        #    (이 리스트 자체는 history에 직접 저장되지 않습니다 — 저장은 아래
        #     new_messages를 통해서만 이뤄집니다. HumanMessage는 RunnableWithMessageHistory가
        #     input_messages_key로 별도 저장하므로 여기서 중복으로 넣지 않습니다.)
        working = prompt_with_history.invoke(payload).to_messages()

        new_messages: list[BaseMessage] = []  # ← 이번 턴에 새로 생긴 메시지만 누적

        # ② model: 첫 호출
        ai_msg = llm_wt.invoke(working)
        working.append(ai_msg)
        new_messages.append(ai_msg)

        # ③ 도구 호출 루프 — tool_calls AIMessage와 ToolMessage를 모두 new_messages에 기록
        while ai_msg.tool_calls:
            for tc in ai_msg.tool_calls:
                tname = tc["name"]
                print(f"  → 도구: {tname} | args: {tc['args']}")
                tool_result = tool_list[tname].invoke(tc)
                working.append(tool_result)
                new_messages.append(tool_result)
            ai_msg = llm_wt.invoke(working)
            working.append(ai_msg)
            new_messages.append(ai_msg)

        # ④ new_messages[-1]은 항상 tool_calls가 없는 최종 AIMessage
        return new_messages

    return RunnableLambda(_run)


# ── 메모리 지원 워크플로우 (RunnableWithMessageHistory 기반) ────────────
def simple_workflow(
    llm,
    question: str,
    tools: list,
    session_id: str = "default",   # ★ 세션 구분자
) -> str:
    """
    질문과 도구 목록을 받아 도구 호출이 완료될 때까지 반복 실행합니다.
    session_id별 대화 이력은 RunnableWithMessageHistory가 자동으로 처리합니다:
      1) get_history(session_id)로 과거 이력을 로드해 chat_history 자리에 주입
         (이전 턴의 tool_calls·ToolMessage까지 그대로 포함되어 있습니다)
      2) _build_agent_runnable이 만든 체인(prompt|model|도구루프) 실행
      3) 반환된 메시지 리스트 전체(HumanMessage 포함)를 자동으로 store에 저장
         → AI 응답뿐 아니라 tool_calls·tool 메시지까지 전부 history에 남습니다.
    """
    agent = _build_agent_runnable(llm, tools)
    chat = RunnableWithMessageHistory(
        agent,
        get_history,
        input_messages_key="input",           # 프롬프트의 human 입력 변수명과 일치
        history_messages_key="chat_history",   # MessagesPlaceholder 변수명과 일치
    )
    cfg = {"configurable": {"session_id": session_id}}   # session_id로 사용자 독립 관리

    print(f"[{session_id}] Q: {question}")
    new_messages = chat.invoke({"input": question}, cfg)

    # ⑤ parser: 최종 AIMessage → 문자열 (호출부에는 기존처럼 문자열만 반환)
    return parser.invoke(new_messages[-1])


In [9]:
# ── 테스트 ─────────────────────────────────────────────────────────────
tools_basic = [get_employee_info, calculate, current_date, web_search]
SESSION = "user_민재"

r1 = simple_workflow(llm, "EMP002 직원 정보 알려줘", tools_basic, session_id=SESSION)
print(f"A: {r1}\n")

r2 = simple_workflow(llm, "1234 * 567은?", tools_basic, session_id=SESSION)
print(f"A: {r2}\n")

r3 = simple_workflow(llm, "오늘 날짜는?", tools_basic, session_id=SESSION)
print(f"A: {r3}\n")

# ★ 이전 대화를 기억하는지 확인
r4 = simple_workflow(llm, "아까 물어봤던 직원 이름이 뭐야?", tools_basic, session_id=SESSION)
print(f"A: {r4}")

r5 = simple_workflow(llm, "오늘 날씨는 어때? 인터넷에서 찾아줘.", tools_basic, session_id=SESSION)
print(f"A: {r5}")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26340\4128944645.py:5: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  r1 = simple_workflow(llm, "EMP002 직원 정보 알려줘", tools_basic, session_id=SESSION)


[user_민재] Q: EMP002 직원 정보 알려줘
  → 도구: get_employee_info | args: {'employee_id': 'EMP002'}
A: EMP002 직원 정보는 다음과 같습니다:

- 이름: 이영희
- 부서: 기획팀
- 직급: PM

[user_민재] Q: 1234 * 567은?


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26340\4128944645.py:8: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  r2 = simple_workflow(llm, "1234 * 567은?", tools_basic, session_id=SESSION)


  → 도구: calculate | args: {'expression': '1234 * 567'}
A: 1234 * 567의 결과는 699,678입니다.

[user_민재] Q: 오늘 날짜는?


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26340\4128944645.py:11: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  r3 = simple_workflow(llm, "오늘 날짜는?", tools_basic, session_id=SESSION)


  → 도구: current_date | args: {}
A: 오늘 날짜는 2026년 9월 11일입니다.

[user_민재] Q: 아까 물어봤던 직원 이름이 뭐야?


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26340\4128944645.py:15: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  r4 = simple_workflow(llm, "아까 물어봤던 직원 이름이 뭐야?", tools_basic, session_id=SESSION)


A: 아까 물어보신 직원의 이름은 이영희입니다.
[user_민재] Q: 오늘 날씨는 어때? 인터넷에서 찾아줘.


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26340\4128944645.py:18: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  r5 = simple_workflow(llm, "오늘 날씨는 어때? 인터넷에서 찾아줘.", tools_basic, session_id=SESSION)


  → 도구: web_search | args: {'query': '오늘 날씨', 'max_results': 5}
A: 오늘 날씨는 다음과 같습니다:

- 서울: 흐림, 21°C
- 수원: 흐림, 20°C
- 인천: 흐림, 22°C
- 대전: 흐림, 21°C
- 광주: 흐림, 24°C
- 부산: 구름 많음, 23°C

전반적으로 흐린 날씨가 지속되고 있습니다.


In [10]:
memory_store

{'user_민재': InMemoryChatMessageHistory(messages=[HumanMessage(content='EMP002 직원 정보 알려줘', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 366, 'total_tokens': 383, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5b6abc5941', 'id': 'chatcmpl-EMlKLyss8kFrUyYK9kSes8E7kZNX4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a08e4d-a4ff-7f20-ad6d-8309c9825e16-0', tool_calls=[{'name': 'get_employee_info', 'args': {'employee_id': 'EMP002'}, 'id': 'call_bPTHmtHBLSQgCMkAgVXQbfKl', 'typ

In [11]:
tools_basic = [get_employee_info, calculate, current_date, web_search]
SESSION = "user_test"

r5 = simple_workflow(llm, "오늘 서울 강남 날씨는 어때? 인터넷에서 찾아줘.", tools_basic, session_id=SESSION)
print(f"A: {r5}")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26340\1308905850.py:4: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  r5 = simple_workflow(llm, "오늘 서울 강남 날씨는 어때? 인터넷에서 찾아줘.", tools_basic, session_id=SESSION)


[user_test] Q: 오늘 서울 강남 날씨는 어때? 인터넷에서 찾아줘.
  → 도구: web_search | args: {'query': '서울 강남 날씨', 'max_results': 5}
A: 서울 강남의 현재 기온은 44도입니다. 더 자세한 내용은 [여기](https://www.youtube.com/watch?v=LJyE54taQKA)에서 확인하실 수 있습니다.


In [12]:
memory_store

{'user_민재': InMemoryChatMessageHistory(messages=[HumanMessage(content='EMP002 직원 정보 알려줘', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 366, 'total_tokens': 383, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5b6abc5941', 'id': 'chatcmpl-EMlKLyss8kFrUyYK9kSes8E7kZNX4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a08e4d-a4ff-7f20-ad6d-8309c9825e16-0', tool_calls=[{'name': 'get_employee_info', 'args': {'employee_id': 'EMP002'}, 'id': 'call_bPTHmtHBLSQgCMkAgVXQbfKl', 'typ